# UC Fibroblasts: Pseudotime, Marker Gene Expression, and TNC Targets

Analyses on connective tissue cells from the UC atlas:

1. **OGN/RSPO3 expression dotplot** across all cell types (Fig. 1H).
2. **Palantir pseudotime** from ADAMDEC1+ Fib root → Crypt Top Fib and Myofibroblast
   terminals (Fig. 1J).
3. **DPT validation** using scanpy diffusion pseudotime with PAGA.
4. **Marker gene dotplots**: pro-inflammatory and pro-fibrotic signatures (Ext. Fig. 5).
5. **TNC downstream targets**: NicheNet-scored target genes displayed as
   dotplot + regulation bar (Fig. 1K / Ext. Fig. 5).

Pericytes are excluded to focus on the stromal fibroblast compartment.

**Inputs** (exported from `03_fibroblast_subclustering.R`):
- `umap_fibroblast_iter4_anno.csv`   – UMAP coordinates
- `con_harmony_iter4.csv`            – Harmony embedding (30 dims)
- `uc_atlas_annotated.h5ad`          – full annotated atlas

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import palantir
from matplotlib.colors import LinearSegmentedColormap

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR   = "/path/to/uc/scrna/output"
FIB_DIR    = f"{DATA_DIR}/fibroblast"
OUTPUT_DIR = f"{DATA_DIR}/fibroblast"

In [ ]:
# Load full atlas
scrna = sc.read_h5ad(f"{DATA_DIR}/uc_atlas_annotated.h5ad")

## 1. OGN / RSPO3 expression across all cell types (Fig. 1H)

In [ ]:
# Exclude rapidly-cycling and transit-amplifying cells for clarity
exclude_general = ["Cycl Myeloid", "Cycl Fib", "SELENOP+ Fib", "Cycl T", "TA"]
scrna_nc = scrna[~scrna.obs["cell_type_general"].isin(exclude_general)].copy()

sc.pl.dotplot(
    scrna_nc,
    var_names=["OGN", "RSPO3"],
    groupby="cell_type_general",
    standard_scale="var",
    colorbar_title="Mean expression\n(scaled)",
    size_title="Fraction\nexpressing",
    figsize=(3, 6),
    save=f"{OUTPUT_DIR}/ogn_rspo3_dotplot.pdf"
)

## 2. Subset fibroblasts and attach embeddings

In [ ]:
fib = scrna[scrna.obs["cell_category"] == "Connective"].copy()
print(fib.obs["cell_type_short"].value_counts())

In [ ]:
# Attach fibroblast UMAP coordinates (exported from R)
umap_fib = pd.read_csv(f"{FIB_DIR}/umap_fibroblast_iter4_anno.csv")
fib.obsm["X_umap"] = umap_fib[["umapharmonyconiter4_1",
                                  "umapharmonyconiter4_2"]].to_numpy()

# Attach Harmony embedding
harm = pd.read_csv(f"{FIB_DIR}/con_harmony_iter4.csv", index_col=0)
fib.obsm["X_harmony"] = harm.to_numpy()

In [ ]:
# Remove pericytes – focus on stromal fibroblast compartment
exclude_fib = ["CD36+ Peri", "RERGL+ Contr Peri"]
fib = fib[~fib.obs["cell_type_short"].isin(exclude_fib)].copy()

sc.pl.umap(fib, color="cell_type_short", frameon=False, s=5)

## 3. Palantir pseudotime (Fig. 1J)

Root: ADAMDEC1+ Fib centroid (most naïve stromal state).

In [ ]:
# Diffusion maps and multi-scale space
dm_res    = palantir.utils.run_diffusion_maps(fib, n_components=5, pca_key="X_harmony")
ms_data   = palantir.utils.determine_multiscale_space(fib)
imputed_X = palantir.utils.run_magic_imputation(fib)

palantir.plot.plot_diffusion_components(fib)
plt.show()

In [ ]:
# Find root cell: closest to ADAMDEC1+ Fib UMAP centroid
coords     = fib.obsm["X_umap"]
early_mask = fib.obs["cell_type_short"] == "ADAMDEC1+ Fib"
early_idx  = np.where(early_mask)[0]
centroid   = coords[early_idx].mean(axis=0)
root_cell  = fib.obs_names[early_idx[np.argmin(
    ((coords[early_idx] - centroid) ** 2).sum(axis=1)
)]]
print(f"Root cell: {root_cell}")

start_states = pd.Series(["ADAMDEC1+ Fib (root)"], index=[root_cell])
palantir.plot.highlight_cells_on_umap(fib, start_states, s=5)
plt.show()

In [ ]:
# Run Palantir from ADAMDEC1+ Fib root
pr_res = palantir.core.run_palantir(fib, root_cell, num_waypoints=500)

palantir.plot.plot_palantir_results(fib, s=3)
plt.show()

## 4. DPT validation (diffusion pseudotime with PAGA)

In [ ]:
fib_cp = fib.copy()

sc.pp.neighbors(fib_cp, use_rep="X_harmony", n_neighbors=20, n_pcs=25)
sc.tl.paga(fib_cp, groups="cell_type_short")
sc.pl.paga(fib_cp, plot=False)
sc.tl.umap(fib_cp, init_pos="paga")

# DPT from ADAMDEC1+ Fib root
fib_cp.uns["iroot"] = np.flatnonzero(
    fib_cp.obs["cell_type_short"] == "ADAMDEC1+ Fib"
)[0]
sc.tl.dpt(fib_cp)

# Restore original UMAP
fib_cp.obsm["X_umap"] = fib.obsm["X_umap"]

In [ ]:
umap_purple = LinearSegmentedColormap.from_list(
    "umap_purple", ["#e8d4cf", "#a16f8d", "#2d223a"]
)
ax = sc.pl.umap(
    fib_cp, color="dpt_pseudotime",
    cmap=umap_purple, vmin=0, vmax="p99",
    na_color="#e8d4cf", frameon=False, show=False
)
ax.set_title("Diffusion Pseudotime (DPT validation)", fontsize=14)
plt.show()

## 5. Marker gene dotplots (Ext. Fig. 5)

Pro-inflammatory and pro-fibrotic gene signatures across fibroblast subtypes.

In [ ]:
# Ordered fibroblast subtypes (ADAMDEC1+ Fib as anchor)
ct_order = [
    "ADAMDEC1+ Fib", "OGN+RSPO3+ Fib", "T-interact Fib",
    "Activ Fib", "CXCL5+ Activ Fib",
    "NRG1+ Crypt Top Fib", "VSTM2A+ Crypt Top Fib",
    "HHIP+ Myofib", "GREM2+ Myofib"
]

# Subset to non-cycling, non-SELENOP+ fibroblasts for clean display
fib_cl = fib[~fib.obs["cell_type_short"].isin(["SELENOP+ Fib", "Cycl Fib"])].copy()
fib_cl.obs["cell_type_short"] = pd.Categorical(
    fib_cl.obs["cell_type_short"], categories=ct_order
)

In [ ]:
pro_inf_genes = ["IL6", "CXCL1", "CXCL2", "CXCL3", "CXCL5", "CCL2",
                 "NFKBIA", "TNFAIP3", "ICAM1", "PTGS2", "MMP3"]
profib_genes  = ["TGFB1", "LGALS3", "COL1A1", "COL1A2",
                 "FN1", "TIMP1", "SERPINE1", "INHBA"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sc.pl.dotplot(
    fib_cl, var_names=pro_inf_genes, groupby="cell_type_short",
    ax=axes[0], show=False, standard_scale=None, vmin=0, vmax=50
)
axes[0].set_title("Pro-inflammatory genes", fontsize=14, fontweight="bold")

sc.pl.dotplot(
    fib_cl, var_names=profib_genes, groupby="cell_type_short",
    ax=axes[1], show=False, standard_scale=None, vmin=0, vmax=50
)
axes[1].set_title("Pro-fibrotic genes", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fibroblast_marker_dotplots.pdf", bbox_inches="tight", dpi=300)
plt.show()

## 6. TNC downstream targets with NicheNet regulation scores (Fig. 1K)

TNC targets identified from NicheNet TF-target network (`13_nichenet.R`).
Expression shown as dotplot; NicheNet regulation score as left-side bar.

In [ ]:
# TNC downstream targets + NicheNet regulation scores (from 13_nichenet.R output)
tnc_reg_df = pd.DataFrame({
    "gene": ["EDNRA", "MMP13", "MMP3", "TIMP3", "ADAMTS4",
              "MBP", "KHDRBS1", "AR", "MMP2", "MMP9", "IL6", "TNF"],
    "tnc_reg": [0.014115, 0.012279, 0.012237, 0.011308, 0.011019,
                0.010472, 0.010289, 0.009902, 0.009145, 0.008327, 0.005794, 0.005686]
}).set_index("gene")

tnc_targets = ["TNC"] + tnc_reg_df.index.tolist()
genes_present = [g for g in tnc_targets if g in fib_cl.var_names]

In [ ]:
# Dotplot with axes swapped (genes on Y) + NicheNet bar on the left
dp = sc.pl.dotplot(
    fib_cl, var_names=genes_present, groupby="cell_type_short",
    standard_scale=None, vmin=0, vmax=30,
    swap_axes=True, return_fig=True, figsize=(5, 4)
)
dp.make_figure()
axes   = dp.get_axes()
main_ax = axes["mainplot_ax"]
fig     = main_ax.figure

# Gene order as displayed in the dotplot
gene_order = [t.get_text() for t in main_ax.get_yticklabels()]
vals = tnc_reg_df.reindex(gene_order)["tnc_reg"]
vals = vals.where(vals.index != "TNC")   # TNC itself has no reg score

# Add regulation bar axis on the left
bbox  = main_ax.get_position()
bar_w = 0.15
bar_ax = fig.add_axes(
    [bbox.x0 - bar_w - 0.16, bbox.y0 - 0.028, bar_w, bbox.height],
    sharey=main_ax
)

y = np.arange(len(gene_order))
bar_ax.barh(y, vals.values, height=0.4, edgecolor="black", linewidth=0.5)

bar_ax.set_xlabel("TNC\nregulation", fontsize=9)
bar_ax.set_xlim(0, np.nanmax(vals.values) * 1.1)
bar_ax.tick_params(axis="y", left=False, labelleft=False)
bar_ax.spines["top"].set_visible(False)
bar_ax.spines["right"].set_visible(False)
bar_ax.spines["left"].set_visible(False)
bar_ax.invert_xaxis()   # bars grow leftward from gene names

main_ax.set_ylim(-0.5, len(gene_order) - 0.5)
main_ax.invert_yaxis()
main_ax.tick_params(axis="y", labelsize=10)
main_ax.set_title("TNC Downstream Targets", fontsize=12, pad=12)

fig.canvas.draw()
plt.savefig(f"{OUTPUT_DIR}/tnc_targets_dotplot_nichenet.pdf", bbox_inches="tight", dpi=300)
plt.show()